# Linear Attention — a toy-scale build of the ungated original

A minimal implementation of **Linear Attention**, from Katharopoulos,
Vyas, Pappas, Fleuret, *"Transformers are RNNs: Fast Autoregressive
Transformers with Linear Attention"* (2020) — the true root of the entire
gated-linear-attention family in this repo (GLA, RetNet, DeltaNet, xLSTM,
KDA all build on top of this exact idea).

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The idea

Softmax attention computes, for every query, a weighted average of all
values, where the weights come from `softmax(q · k)`. The softmax is what
makes this expensive: it doesn't factor into separate query-only and
key-only pieces, so you can't avoid comparing every query against every key.

Linear attention's core trick: **replace the softmax with a positive kernel
feature map** `phi`, and drop the exponential entirely:

```
attention_weight(q_t, k_s) ~ phi(q_t) . phi(k_s)     (instead of exp(q_t . k_s))
```

Because this new "attention weight" factors into a `phi(q_t)` piece and a
`phi(k_s)` piece multiplied together, the whole computation can be
rearranged: instead of comparing the query against every key one at a time,
you can accumulate `phi(k_s) (x) v_s` into a running sum as you go, and read
it out with `phi(q_t)` at the end. That's exactly the "state that gets
updated one token at a time" pattern used by every other architecture in
this repo — this is where the pattern originates.

```
S_t = S_{t-1} + phi(k_t) (x) v_t         # just accumulate -- no decay, no erase, nothing
Z_t = Z_{t-1} + phi(k_t)                  # running normalizer
o_t = (phi(q_t)^T S_t) / (phi(q_t)^T Z_t)   # normalized readout
```

This notebook uses `phi(x) = elu(x) + 1` (the feature map from the paper),
which guarantees every "attention weight" stays non-negative, mimicking one
property of the softmax without needing the softmax itself.

## 2. Why every other notebook in this repo adds something on top

Notice there's no decay, no erase step, no gating of any kind — the state
just accumulates every key-value pair it's ever seen, forever, at equal
weight. This works, but it has a real cost: with nothing to forget, the
state gets muddier and muddier as the sequence gets longer, since old and
new associations for similar keys all get blended together indiscriminately
by the `+=`.

This turns out to be the central limitation the rest of this repo's
recurrent-state architectures are built to fix:

- **GLA** (`../gla`) adds a per-channel forget gate, so old information can
  fade.
- **DeltaNet** (`../deltanet`) adds the delta rule, so writing a new value
  for a key first erases the old value for that same key, rather than
  blending them.
- **KDA** (`../kda`), **RetNet** (`../retnet`), and **xLSTM** (`../xlstm`)
  each combine decay and/or the delta rule in their own way.

Reading this notebook first, then any of those, should make it much clearer
exactly what problem each one is solving.

> **Simplification used here:** none, really — this is already about as
> simple as the mechanism gets. The one choice worth noting is the feature
> map itself: `elu(x)+1` is the paper's original choice, but plenty of later
> work (including some cited by GLA and DeltaNet) experiments with other
> feature maps, including learned ones.

In [ ]:
class LinearAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32):
        super().__init__()
        self.h, self.dh = n_heads, d_head
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.out_proj = nn.Linear(inner, d_model, bias=False)

    def feature_map(self, x):
        return F.elu(x) + 1.0     # ensures positivity, so "attention weights" stay >= 0

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.h, self.dh
        q = self.feature_map(self.q_proj(x).view(B, T, H, Dh))
        k = self.feature_map(self.k_proj(x).view(B, T, H, Dh))
        v = self.v_proj(x).view(B, T, H, Dh)

        S = x.new_zeros(B, H, Dh, Dh)     # state: running sum of phi(k_s) (x) v_s
        Z = x.new_zeros(B, H, Dh)          # normalizer: running sum of phi(k_s)
        outs = []
        for t in range(T):
            k_t, v_t, q_t = k[:, t], v[:, t], q[:, t]
            S = S + k_t.unsqueeze(-1) * v_t.unsqueeze(-2)     # accumulate -- that's the entire update
            Z = Z + k_t
            num = torch.einsum('bhd,bhde->bhe', q_t, S)
            den = torch.einsum('bhd,bhd->bh', q_t, Z).unsqueeze(-1).clamp_min(1e-6)
            outs.append(num / den)
        o = torch.stack(outs, dim=1).reshape(B, T, H * Dh)
        return self.out_proj(o)

## 3. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([LinearAttention(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through Linear Attention once it's wired into a real model. So the rest of this
notebook:

1. wraps Linear Attention into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Watch it struggle on a task that needs forgetting.** The toy periodic
  task here is short and repetitive enough that plain accumulation still
  works fine. Try a longer, less repetitive sequence and compare against
  `../gla` (same model, plus a forget gate) — this is the cleanest way to
  actually *feel* why every other notebook in this repo bothers with
  gating.
- **Try a different feature map.** Swap `elu(x)+1` for something else
  positive (e.g. `softmax` over a small random-features expansion, as in
  the Performer paper) and see how it changes what the toy model learns.
- **Read `../deltanet` and `../gla` next** — both add exactly one thing on
  top of this notebook's bare recurrence.

Reference: Katharopoulos, Vyas, Pappas, Fleuret, *"Transformers are RNNs:
Fast Autoregressive Transformers with Linear Attention,"* 2020.